In [13]:
from discreteNPIV.simulate import *
alpha = find_alpha(190, 200)
print(alpha)

0.20166337490081787


In [1]:
import os
import itertools
import pandas as pd

# Define parameter grid
params = {
    'n': [ 10, 20, 30, 50, 100, 500],
    'd': [300],
    'K_sparsity': [(100, 25), (100, 50), (100, 75), (100, 100), (100, 125), (100, 150), (100, 175), (100, 190), (100, 200)]
}
params = {
    'n': [10, 50, 100, 200, 4],
    'd': [200],
    'K_sparsity': [(100, 10), (100, 25), (100, 50), (100, 75), (100, 100), (100, 125), (100, 150), (100, 175), (100, 190), (100, 200)],
    'oracle': [False, True]
}

# Generate all combinations of parameters
param_combinations = list(itertools.product(params['n'], params['d'], params['K_sparsity'], params['oracle']))

# Collect data frames
dfs = []

# Load data for each parameter setting
for n, d, (K, sparsity), oracle in param_combinations:
    filename = f"results/sim_n{n}_d{d}_K{K}_sparsity{sparsity}_oracle_{oracle}_final.csv"
    
    if os.path.exists(filename):  # Check if file exists before loading
        df = pd.read_csv(filename)
        
        # Add parameters as columns
        df["n"] = n
        df["d"] = d
        df["K"] = K
        df["sparsity"] = sparsity
        df["oracle"] = oracle
        
        dfs.append(df)

# Combine all data frames if any files were found
combined_df = pd.concat(dfs, ignore_index=True)
combined_df.to_csv("results/combined_results.csv", index=False)
 

In [9]:
df 

,n,d,K,alpha,sparsity,Bias_Plugin,Bias_IPW,Bias_IPW_No_Cross,Bias_Debiased,Bias_Debiased_No_Cross,...,MSE_Plugin,MSE_IPW,MSE_IPW_No_Cross,MSE_Debiased,MSE_Debiased_No_Cross,MSE_Debiased_Single_Fold,Coverage_Probability,Coverage_Probability_No_Cross,Coverage_Probability_Single_Fold,oracle
1,10,200,100,1.224955,10,0.002918,-0.013302,0.015706,-0.005605,-0.002995,...,0.003605,0.143528,0.095770,0.003208,0.004493,0.006794,0.945,0.965,0.950,True
3,10,200,100,0.953795,25,-0.002886,0.035115,-0.034551,0.002719,-0.008208,...,0.002783,0.149428,0.118984,0.003277,0.006224,0.006388,0.950,0.940,0.935,True
5,10,200,100,0.772986,50,-0.000165,-0.024003,0.028156,-0.005948,-0.004608,...,0.002578,0.074953,0.129373,0.002826,0.005893,0.004753,0.925,0.950,0.960,True
7,10,200,100,0.662580,75,0.001462,-0.043507,0.019964,-0.000280,-0.003416,...,0.002986,0.099329,0.110491,0.002887,0.005775,0.005189,0.930,0.945,0.955,True
9,10,200,100,0.574289,100,0.000416,0.022401,-0.021952,-0.000195,-0.002806,...,0.005490,0.135434,0.123125,0.005497,0.008269,0.008670,0.775,0.875,0.865,True
11,10,200,100,0.492585,125,0.010614,-0.010653,-0.009905,0.002723,0.011427,...,0.012650,0.120351,0.186015,0.012174,0.017460,0.014084,0.600,0.655,0.730,True
13,10,200,100,0.406893,150,-0.009713,-0.010182,-0.063567,-0.014870,-0.017701,...,0.032311,0.224243,0.233843,0.032137,0.047771,0.034952,0.370,0.560,0.500,True
15,10,200,100,0.300249,175,-0.037331,0.061599,-0.009306,-0.043862,-0.032356,...,0.213448,1.325805,0.530965,0.206103,0.244567,0.205027,0.165,0.300,0.235,True
21,50,200,100,1.224955,10,-0.004192,-0.000516,-0.007150,-0.012753,-0.002081,...,0.003760,0.091468,0.017655,0.019395,0.001484,0.027091,0.950,0.970,0.920,True
23,50,200,100,0.953795,25,0.004166,0.057445,-0.015774,0.001876,-0.001781,...,0.006114,0.156029,0.016714,0.008072,0.002159,0.019673,0.935,0.945,0.945,True


In [3]:
df = combined_df[combined_df['oracle'] == True]



In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Apply filters correctly
df = combined_df[(combined_df['n'] != 4) & (combined_df['oracle'] == False) & (combined_df['sparsity'] <= 175)]

# Define RMSE method mappings (Omitting IPW)
rmse_method_mapping = {
   # "IPW": ["MSE_IPW"],
   # "Plug-in": ["MSE_Plugin"],
    "DML (JIVE)": ["MSE_Debiased"],
    "DML (2SLS)": ["MSE_Debiased_No_Cross"],
    "DML (JIVE 1-fold)": ["MSE_Debiased_Single_Fold"],
}

# Melt the DataFrame for RMSE
df_rmse_long = pd.melt(
    df,
    id_vars=["n", "d", "K", "alpha", "sparsity"],
    value_vars=[col for cols in rmse_method_mapping.values() for col in cols],
    var_name="metric",
    value_name="value",
)

# Assign RMSE methods
df_rmse_long["method"] = df_rmse_long["metric"].map(
    {col: method for method, cols in rmse_method_mapping.items() for col in cols}
)

# Convert MSE to RMSE
df_rmse_long["value"] = np.sqrt(df_rmse_long["value"])
df_rmse_long["metric"] = "RMSE"  # Rename for clarity

# Get unique sample sizes
n_values = sorted(df_rmse_long["n"].unique())
num_n = len(n_values)

# Create subplots for RMSE
fig, axes = plt.subplots(1, num_n, figsize=(4 * num_n, 4), sharey=True)

# Store lines for the shared legend
lines, labels = [], []

for col, n_value in enumerate(n_values):
    ax = axes[col] if num_n > 1 else axes  # Handle single-panel case
    
    subset = df_rmse_long[df_rmse_long["n"] == n_value]
    
    # Get unique methods
    methods = subset["method"].dropna().unique()

    # Plot each method separately
    for method in methods:
        df_method = subset[subset["method"] == method].sort_values("sparsity")
        if not df_method.empty:
            line, = ax.plot(df_method["sparsity"], df_method["value"], label=method, marker='o', linestyle='-')
            if col == 0:  # Store legend items only once
                lines.append(line)
                labels.append(method)

    ax.set_title(f"RMSE (n={n_value})")
    ax.set_xlabel("Sparsity")
    if col == 0:
        ax.set_ylabel("RMSE")
    ax.grid(True)

# Add shared legend outside of the plot
fig.legend(lines, labels, title="Method", loc="upper center", bbox_to_anchor=(0.5, 1.1), ncol=3)

plt.tight_layout()
plt.savefig('RMSE_plot_2sls.png', bbox_inches='tight', dpi=300)
plt.close()



# Define Coverage method mappings
coverage_methods = {
    "DML (JIVE)": "Coverage_Probability",
    "DML (2SLS)": "Coverage_Probability_No_Cross",
    "DML (JIVE 1-fold)": "Coverage_Probability_Single_Fold",
}

# Melt the DataFrame for Coverage
df_coverage_long = pd.melt(
    df,
    id_vars=["n", "d", "K", "alpha", "sparsity"],
    value_vars=list(coverage_methods.values()),
    var_name="metric",
    value_name="value",
)

# Assign Coverage methods
df_coverage_long["method"] = df_coverage_long["metric"].map(
    {col: method for method, col in coverage_methods.items()}
)
df_coverage_long["metric"] = "Coverage"  # Rename for clarity

# Create subplots for Coverage
fig, axes = plt.subplots(1, num_n, figsize=(4 * num_n, 4), sharey=True)

# Store lines for the shared legend
lines, labels = [], []

for col, n_value in enumerate(n_values):
    ax = axes[col] if num_n > 1 else axes  # Handle single-panel case
    
    subset = df_coverage_long[df_coverage_long["n"] == n_value]
    
    # Get unique methods
    methods = subset["method"].dropna().unique()

    # Plot each method separately
    for method in methods:
        df_method = subset[subset["method"] == method].sort_values("sparsity")
        if not df_method.empty:
            line, = ax.plot(df_method["sparsity"], df_method["value"], label=method, marker='o', linestyle='-')
            if col == 0:  # Store legend items only once
                lines.append(line)
                labels.append(method)

    # Add the 95% reference line
    ax.axhline(y=0.95, color="grey", linestyle="dashed", linewidth=1)

    ax.set_title(f"Coverage (n={n_value})")
    ax.set_xlabel("Sparsity")
    if col == 0:
        ax.set_ylabel("Coverage")
    ax.grid(True)

# Add shared legend outside of the plot
fig.legend(lines, labels, title="Method", loc="upper center", bbox_to_anchor=(0.5, 1.1), ncol=3)

plt.tight_layout()
plt.savefig('Coverage_plot_2sls.png', bbox_inches='tight', dpi=300)
plt.close()
